In [1]:
import re
from camel_tools.tokenizers.word import simple_word_tokenize
from camel_tools.tokenizers.morphological import MorphologicalTokenizer
from camel_tools.disambig.mle import MLEDisambiguator
from farasa.stemmer import FarasaStemmer
from farasa.segmenter import FarasaSegmenter
from camel_tools.morphology.database import MorphologyDB
from camel_tools.morphology.analyzer import Analyzer
import pyarabic.araby as araby

In [2]:
import os
import re
from farasa.stemmer import FarasaStemmer
from farasa.segmenter import FarasaSegmenter
from camel_tools.disambig.mle import MLEDisambiguator
from camel_tools.morphology.database import MorphologyDB
from camel_tools.morphology.analyzer import Analyzer
from camel_tools.tokenizers.word import simple_word_tokenize
from camel_tools.tokenizers.morphological import MorphologicalTokenizer

In [ ]:
def normalize_arabic(text):
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ؤ", "ء", text)
    text = re.sub("ئ", "ء", text)
    text = re.sub("ة", "ه", text)
    text = re.sub("گ", "ك", text)
    text = re.sub("ڤ", "ف", text)
    text = re.sub("چ", "ج", text)
    text = re.sub("پ", "ب", text)
    text = re.sub("ڜ", "ش", text)
    text = re.sub("ڪ", "ك", text)
    text = re.sub("ڧ", "ق", text)
    text = re.sub("ٱ", "ا", text)
    return text

In [ ]:
def remove_arabic_noise(text):
    # Remove diacritics
    text = re.sub(r'[\u0617-\u061A\u064B-\u0652]', '', text)
    # Remove tatweel
    text = re.sub(r'\u0640', '', text)
    # Remove non-Arabic characters
    text = re.sub(r'[^\u0600-\u06FF\s]', '', text)
    # Remove HTML tags
    text = re.sub('<.*?>', '', text)
    # Remove extra whitespaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [15]:
class ArabicTokenizer:
    def __init__(self):
        self.farasa = FarasaSegmenter(interactive=True)
        self.mle_msa = MLEDisambiguator.pretrained('calima-msa-r13')
        self.morph_tokenizer = MorphologicalTokenizer(disambiguator=self.mle_msa, scheme='bwtok')

    def tokenize_words(self, text, method='simple'):
        """Tokenizes Arabic text into words."""
        if method == 'simple':
            return simple_word_tokenize(text)
        elif method == 'morphological':
            words = simple_word_tokenize(text)
            return self.morph_tokenizer.tokenize(words)
        else:
            raise ValueError("Invalid method. Use 'simple' or 'morphological'")

    def tokenize_sentences(self, text):
        """Splits Arabic text into sentences using Farasa."""
        return self.farasa.segment(text).split('.')

    def tokenize_text(self, text, word_method='morphological'):
        """Tokenizes text into sentences, then words."""
        sentences = self.tokenize_sentences(text)
        return [self.tokenize_words(sent.strip(), word_method) for sent in sentences if sent.strip()]

# Usage example
tokenizer = ArabicTokenizer()
text = "مرحبا. كيف حالك؟"
result = tokenizer.tokenize_text(text)
print(result)

[2024-12-31 16:11:06,021 - farasapy_logger - WARNING]: Be careful with large lines as they may break on interactive mode. You may switch to Standalone mode for such cases.


[['مرحب', '', 'إ'], ['كيف', 'حال', '', 'ك', '؟']]


In [3]:
class ArabicTextProcessor:
    def __init__(self):
        # Initialize Farasa tools
        self.farasa_stemmer = FarasaStemmer()
        self.farasa_segmenter = FarasaSegmenter(interactive=True)
        
        # Initialize CAMeL Tools
        self.db = MorphologyDB.builtin_db()
        self.analyzer = Analyzer(self.db)
        self.mle_msa = MLEDisambiguator.pretrained('calima-msa-r13')
        self.morph_tokenizer = MorphologicalTokenizer(disambiguator=self.mle_msa, scheme='bwtok')

    def stem(self, text):
        """Stems Arabic text using Farasa."""
        return self.farasa_stemmer.stem(text)

    def tokenize_words(self, text, method='simple'):
        """Tokenizes Arabic text into words."""
        if method == 'simple':
            return simple_word_tokenize(text)
        elif method == 'morphological':
            words = simple_word_tokenize(text)
            return self.morph_tokenizer.tokenize(words)
        else:
            raise ValueError("Invalid method. Use 'simple' or 'morphological'")

    def tokenize_sentences(self, text):
        """Splits Arabic text into sentences using Farasa."""
        return self.farasa_segmenter.segment(text).split('.')

    def tokenize_text(self, text, word_method='morphological'):
        """Tokenizes text into sentences, then words."""
        sentences = self.tokenize_sentences(text)
        return [self.tokenize_words(sent.strip(), word_method) for sent in sentences if sent.strip()]
    
    def handle_diacritics(text, method='remove'):
        if method == 'remove':
            return araby.strip_diacritics(text)
        elif method == 'keep':
            return text
        elif method == 'normalize':
            return araby.normalize_hamza(araby.strip_shadda(text))
        
    def handle_numbers_and_special_chars(text, mode='remove'):
        if mode == 'remove':
            # Remove numbers and special characters
            return re.sub(r'[^\u0600-\u06FF\s]', '', text)
        elif mode == 'normalize':
            # Normalize Arabic numbers to Hindi numbers
            number_map = {
                '٠': '0', '١': '1', '٢': '2', '٣': '3', '٤': '4',
                '٥': '5', '٦': '6', '٧': '7', '٨': '8', '٩': '9'
            }
            for arabic, hindi in number_map.items():
                text = text.replace(arabic, hindi)
            return text

    def process_text(self, text, operations=None):
        """Process text with multiple operations.
        
        Args:
            text (str): Input Arabic text
            operations (list): List of operations to perform ('stem', 'tokenize_words', 'tokenize_sentences')
        """
        if operations is None:
            operations = ['stem']
        
        results = {}
        for op in operations:
            if op == 'stem':
                results['stemmed'] = self.stem(text)
            elif op == 'tokenize_words':
                results['words'] = self.tokenize_words(text)
            elif op == 'tokenize_sentences':
                results['sentences'] = self.tokenize_sentences(text)
        
        return results

In [4]:
# Example usage
if __name__ == "__main__":
    processor = ArabicTextProcessor()
    text = "الكتب المدرسية مفيدة للطلاب. مرحبا، كيف حالك؟"
    
    # Process with multiple operations
    results = processor.process_text(text, ['stem', 'tokenize_sentences', 'tokenize_words'])
    print("Results:", results)
    
    # Full text tokenization
    tokenized = processor.tokenize_text(text)
    print("Tokenized text:", tokenized)

[2024-12-31 20:15:02,387 - farasapy_logger - WARNING]: Be careful with large lines as they may break on interactive mode. You may switch to Standalone mode for such cases.


Results: {'stemmed': 'كتاب مدرسي مفيد طالب . مرحبا ، كيف حال ؟', 'sentences': ['ال+كتب ال+مدرسي+ة مفيد+ة ل+ال+طلاب ', ' مرحب+ا ، كيف حال+ك ؟'], 'words': ['الكتب', 'المدرسية', 'مفيدة', 'للطلاب', '.', 'مرحبا', '،', 'كيف', 'حالك', '؟']}
Tokenized text: [['ال', '', 'كتب', 'ال', '', 'مدرس_+ي_+ي', '', 'ه', 'مفيد', '', 'ه', 'ل', '', 'ال', '', 'طلاب'], ['مرحب', '', 'إ', '،', 'كيف', 'حال', '', 'ك', '؟']]


In [23]:
# Example usage
#text_with_diacritics = "اللُّغَةُ العَرَبِيَّةُ جَمِيلَةٌ"
#removed_diacritics = handle_diacritics(text_with_diacritics, 'remove')
#normalized_diacritics = handle_diacritics(text_with_diacritics, 'normalize')

#print("Original:", text_with_diacritics)
#print("Removed diacritics:", removed_diacritics)
#print("Normalized diacritics:", normalized_diacritics)

In [25]:
# Example usage
#text = "يوجد ٣ تفاحات و٥ برتقالات في السلة!"
#removed_numbers = handle_numbers_and_special_chars(text, 'remove')
#normalized_numbers = handle_numbers_and_special_chars(text, 'normalize')

#print("Original:", text)
#print("Removed numbers and special chars:", removed_numbers)
#print("Normalized numbers:", normalized_numbers)

In [3]:


class ArabicTextProcessor:
    def __init__(self):
        # Initialize Farasa tools
        self.farasa_stemmer = FarasaStemmer()
        self.farasa_segmenter = FarasaSegmenter(interactive=True)
        
        # Initialize CAMeL Tools
        self.db = MorphologyDB.builtin_db()
        self.analyzer = Analyzer(self.db)
        self.mle_msa = MLEDisambiguator.pretrained('calima-msa-r13')
        self.morph_tokenizer = MorphologicalTokenizer(disambiguator=self.mle_msa, scheme='bwtok')

    def stem(self, text):
        """Stems Arabic text using Farasa."""
        return self.farasa_stemmer.stem(text)

    def tokenize_words(self, text, method='simple'):
        """Tokenizes Arabic text into words."""
        if method == 'simple':
            return simple_word_tokenize(text)
        elif method == 'morphological':
            words = simple_word_tokenize(text)
            return self.morph_tokenizer.tokenize(words)
        else:
            raise ValueError("Invalid method. Use 'simple' or 'morphological'")

    def tokenize_sentences(self, text):
        """Splits Arabic text into sentences using Farasa."""
        return self.farasa_segmenter.segment(text).split('.')

    def tokenize_text(self, text, word_method='morphological'):
        """Tokenizes text into sentences, then words."""
        sentences = self.tokenize_sentences(text)
        return [self.tokenize_words(sent.strip(), word_method) for sent in sentences if sent.strip()]
    
    def handle_diacritics(self, text, method='remove'):
        if method == 'remove':
            return araby.strip_diacritics(text)
        elif method == 'keep':
            return text
        elif method == 'normalize':
            return araby.normalize_hamza(araby.strip_shadda(text))
        
    def handle_numbers_and_special_chars(self, text, mode='remove'):
        if mode == 'remove':
            # Remove numbers and special characters
            return re.sub(r'[^\u0600-\u06FF\s]', '', text)
        elif mode == 'normalize':
            # Normalize Arabic numbers to Hindi numbers
            number_map = {
                '٠': '0', '١': '1', '٢': '2', '٣': '3', '٤': '4',
                '٥': '5', '٦': '6', '٧': '7', '٨': '8', '٩': '9'
            }
            for arabic, hindi in number_map.items():
                text = text.replace(arabic, hindi)
            return text

    def process_text(self, text, operations=None):
        """Process text with multiple operations.
        
        Args:
            text (str): Input Arabic text
            operations (list): List of operations to perform ('stem', 'tokenize_words', 'tokenize_sentences')
        """
        if operations is None:
            operations = ['stem']
        
        results = {}
        for op in operations:
            if op == 'stem':
                results['stemmed'] = self.stem(text)
            elif op == 'tokenize_words':
                results['words'] = self.tokenize_words(text)
            elif op == 'tokenize_sentences':
                results['sentences'] = self.tokenize_sentences(text)
        
        return results

# Directory containing the files
dir_path = "Scripts"
processor = ArabicTextProcessor()

[2025-01-01 02:53:02,115 - farasapy_logger - WARNING]: Be careful with large lines as they may break on interactive mode. You may switch to Standalone mode for such cases.


In [ ]:


# Process each file
for root, dirs, files in os.walk(dir_path):
    for file_name in files:
        # Skip irrelevant files and specifically log.txt
        if file_name == 'log.txt':
            continue
        if file_name.endswith('.txt') or file_name.endswith('.py') or file_name.endswith('.ipynb'):
            file_path = os.path.join(root, file_name)
            
            with open(file_path, 'r', encoding='utf-8') as file:
                content = file.read()

            # Process the text
            processed_text = processor.tokenize_text(content)

            # Save the processed text to a new file
            output_path = os.path.join(root, f'processed_tokenized{file_name}')
            with open(output_path, 'w', encoding='utf-8') as output_file:
                output_file.write(str(processed_text))

[2025-01-01 02:52:25,979 - farasapy_logger - WARNING]: Be careful with large lines as they may break on interactive mode. You may switch to Standalone mode for such cases.


KeyboardInterrupt: 

Exception ignored in: 'zmq.backend.cython._zmq.Frame.__del__'
Traceback (most recent call last):
  File "_zmq.py", line 160, in zmq.backend.cython._zmq._check_rc
KeyboardInterrupt: 


In [4]:

# Specify the directory containing the files to process
dir_path = 'Scripts'

def process_text(content):
    # Step 1: Handle diacritics (remove them)
    content = processor.handle_diacritics(content, method='normalize')
    
    # Step 2: Handle numbers and special characters (remove them)
    content = processor.handle_numbers_and_special_chars(content, mode='normalize')
    
    # Step 3: Apply stemming
    content = processor.stem(content)
    
    # Step 4: Remove unwanted items like '[', 'موسيقى', ']'
    unwanted_items = '[', 'موسيقى', ']'
    content = eval(content)  # Convert string to list if it's serialized like a list
    content = [item for item in content if item not in unwanted_items]
    
    return str(content)  # Convert back to string if needed


# Process only files starting with 'processed_tokenized'
for root, dirs, files in os.walk(dir_path):
    for file_name in files:
        # Skip log.txt and unrelated files
        if file_name == 'log.txt' or not file_name.startswith('processed_tokenized'):
            continue
        
        # Process only files with the specified extensions
        if file_name.endswith('.txt') or file_name.endswith('.py') or file_name.endswith('.ipynb'):
            file_path = os.path.join(root, file_name)
            
            # Read the file content
            with open(file_path, 'r', encoding='utf-8') as file:
                content = file.read()
            
            # Apply the processing
            processed_content = process_text(content)
            
            # Create the corresponding output folder structure
            relative_path = os.path.relpath(root, dir_path)
            processed_folder = os.path.join(dir_path, 'processed', relative_path)
            os.makedirs(processed_folder, exist_ok=True)
            
            # Save the processed content to the processed folder
            output_path = os.path.join(processed_folder, f'final_{file_name}')
            with open(output_path, 'w', encoding='utf-8') as output_file:
                output_file.write(processed_content)

print(f"Processing complete. Final processed files are saved inside the 'processed' folder within the original directory structure.")


Processing complete. Final processed files are saved inside the 'processed' folder within the original directory structure.


In [5]:
import os
import ast  # To safely parse string representation of tokenized data

def clean_tokenized_data(tokenized_data):
    # Define unwanted tokens or patterns
    unwanted_tokens = ['[', ']', '_', ' ']
    
    # Function to clean a single token
    def clean_token(token):
        # Strip leading/trailing spaces and remove unwanted characters
        return token.strip().replace('_', '').replace('[', '').replace(']', '')
    
    # Apply cleaning to each token in the data
    cleaned_data = []
    for sentence in tokenized_data:
        cleaned_sentence = [
            clean_token(token) for token in sentence if token.strip() not in unwanted_tokens
        ]
        # Remove empty tokens
        cleaned_sentence = [t for t in cleaned_sentence if t]
        cleaned_data.append(cleaned_sentence)
    
    return cleaned_data

# Path to the processed folder
processed_dir = os.path.join('Scripts', 'processed')

# Traverse through the processed folder
for root, dirs, files in os.walk(processed_dir):
    for file_name in files:
        if file_name.endswith('.txt'):  # Process only text files
            file_path = os.path.join(root, file_name)
            
            # Read the file content
            with open(file_path, 'r', encoding='utf-8') as file:
                content = file.read()
            
            # Parse the tokenized data safely
            try:
                tokenized_data = ast.literal_eval(content)  # Assumes content is a list of lists
                if not isinstance(tokenized_data, list):
                    raise ValueError("Data is not in the expected tokenized format.")
            except Exception as e:
                print(f"Skipping file {file_name}: {e}")
                continue
            
            # Clean the tokenized data
            cleaned_data = clean_tokenized_data(tokenized_data)
            
            # Save the cleaned data back to the file (overwrite)
            with open(file_path, 'w', encoding='utf-8') as file:
                file.write(str(cleaned_data))

print("Tokenized data cleaned and saved.")


Tokenized data cleaned and saved.


In [6]:
import os
import ast  # To safely parse string representation of tokenized data

def clean_tokenized_data(tokenized_data):
    # Define unwanted tokens
    unwanted_tokens = {'موسيقى', 'ضحك'}
    
    # Function to clean a single token
    def clean_token(token):
        # Strip leading/trailing spaces and check if token is unwanted
        cleaned_token = token.strip()
        return None if cleaned_token in unwanted_tokens else cleaned_token
    
    # Apply cleaning to each token in the data
    cleaned_data = []
    for sentence in tokenized_data:
        cleaned_sentence = [
            clean_token(token) for token in sentence
        ]
        # Remove None tokens and empty sentences
        cleaned_sentence = [t for t in cleaned_sentence if t]
        if cleaned_sentence:  # Only add non-empty sentences
            cleaned_data.append(cleaned_sentence)
    
    return cleaned_data

# Paths to input and output folders
processed_dir = os.path.join('Scripts', 'processed')
new_folder = os.path.join('Scripts', 'cleaned')
os.makedirs(new_folder, exist_ok=True)  # Create new folder if it doesn't exist

# Traverse through the processed folder
for root, dirs, files in os.walk(processed_dir):
    for file_name in files:
        if file_name.endswith('.txt'):  # Process only text files
            file_path = os.path.join(root, file_name)
            
            # Read the file content
            with open(file_path, 'r', encoding='utf-8') as file:
                content = file.read()
            
            # Parse the tokenized data safely
            try:
                tokenized_data = ast.literal_eval(content)  # Assumes content is a list of lists
                if not isinstance(tokenized_data, list):
                    raise ValueError("Data is not in the expected tokenized format.")
            except Exception as e:
                print(f"Skipping file {file_name}: {e}")
                continue
            
            # Clean the tokenized data
            cleaned_data = clean_tokenized_data(tokenized_data)
            
            # Save the cleaned data to the new folder
            relative_path = os.path.relpath(root, processed_dir)
            cleaned_folder = os.path.join(new_folder, relative_path)
            os.makedirs(cleaned_folder, exist_ok=True)
            
            output_path = os.path.join(cleaned_folder, file_name)
            with open(output_path, 'w', encoding='utf-8') as file:
                file.write(str(cleaned_data))

print("Unwanted tokens removed and cleaned data saved to the new folder.")


Unwanted tokens removed and cleaned data saved to the new folder.


In [7]:

# Drop all files created
for root, dirs, files in os.walk(dir_path):
    for file_name in files:
        if file_name.startswith('processed_'):  # Check for processed files
            file_path = os.path.join(root, file_name)
            os.remove(file_path)
            print(f"Deleted: {file_path}")

print("Text processing complete. All processed files have been deleted.")


Deleted: Scripts\1السنافرyoutube1\processed_log.txt
Deleted: Scripts\_توم_و_جيريyoutube1\processed_log.txt
Deleted: Scripts\_لغز_مارتنyoutube1\processed_[1R0S4KTbhr0] - لغز مارتن _ S1EP18 _ عودة الكاهن المظلم.txt
Deleted: Scripts\أبطال_الكرة1youtube1\processed_log.txt
Deleted: Scripts\أبطال_الكرة_21youtube1\processed_log.txt
Deleted: Scripts\أبطال_الكرة_31youtube1\processed_log.txt
Deleted: Scripts\أبطال_النينجاyoutube1\processed_logg.txt
Deleted: Scripts\أبناء_الوطنyoutube1\processed_log.txt
Deleted: Scripts\أرض_السعادةyoutube1\processed_log.txt
Deleted: Scripts\أرغاي_الفارس_النبيلyoutube1\processed_log.txt
Deleted: Scripts\أسرار_الغابةyoutube1\processed_log.txt
Deleted: Scripts\أسرار_المحيطyoutube1\processed_[aXhNgVQyf24] - أنمي أسرار المحيط الحلقة 2 مدبلج عربي.txt
Deleted: Scripts\أكاديمية_الشرطةyoutube1\processed_log.txt
Deleted: Scripts\أنا_و_أخيyoutube1\processed_logg.txt
Deleted: Scripts\ابل_وأنيونyoutube1\processed_log.txt
Deleted: Scripts\البؤساءyoutube1\processed_[0VpxFkA2jW8